In [1]:
!pip install mysql-connector-python

     ---------------------------------------- 16.4/16.4 MB 9.8 MB/s eta 0:00:00



[notice] A new release of pip available: 22.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np

In [2]:
url = "https://raw.githubusercontent.com/JDELGADO2607/proyecto_ingenieria_de_datos_equipo4/refs/heads/Juan_Delgado/20250824_-_Export_Combined_Table_(ICE-Factset-EDI).csv" 

data = pd.read_csv(url, delimiter = ';', encoding = 'latin-1')


C:\Users\juanm\AppData\Local\Temp\ipykernel_17196\352481037.py:3: DtypeWarning: Columns (1,19,30) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(url, delimiter = ';', encoding = 'latin-1')


In [3]:
import mysql.connector

conexion = mysql.connector.connect(
    host = 'localhost',
    user = 'root',
    database = 'solactive_price_validation'
)

In [4]:
cursor = conexion.cursor()

In [5]:
raw_data = data[['rt_close', 
                'date', 
                'rt_ts', 
                'rt_currency', 
                'region', 
                'mic', 
                'factset_close', 
                'factset_late_close', 
                'factset_currency', 
                'factset_late_currency', 
                'edi_close', 
                'edi_currency',
                'real_asset_class',
                'status']]

raw_data.head()

,rt_close,date,rt_ts,rt_currency,region,mic,factset_close,factset_late_close,factset_currency,factset_late_currency,edi_close,edi_currency,real_asset_class,status
0,0.337,14/08/2025,2025-08-14 10:00:02.319749 UTC,KWD,ASIA,XKUW,0.337,0.337,KWD,KWD,0.337,KWD,SHARE,VALIDATED
1,0.9318,6/01/2025,2025-01-06 17:00:01.219813 UTC,EUR,EUROPE,XAMS,0.9318,0.9318,EUR,EUR,0.9309,EUR,SHARE,UNVALIDATED
2,3140,17/06/2025,2025-06-17 20:02:07.214827 UTC,CLP,AMERICA,XSGO,3140,NaN,CLP,CLP,3140,CLP,SHARE,VALIDATED
3,116,14/03/2025,2025-03-14 16:20:01.296614 UTC,DKK,EUROPE,XCSE,116,116,DKK,DKK,116,DKK,SHARE,VALIDATED
4,97.72,31/12/2024,NaN,EUR,EUROPE,XAMS,97.72,97.72,EUR,EUR,97.72,EUR,SHARE,VALIDATED


In [8]:
lista_reglas = []

#Para cada regla aplicable se deberá crear una funcion que la ejecute y deberá realizarse un .append a lista_reglas y cada función deberá devolver un valor entre "True" y "False" 

def aplicar_reglas(raw_data: pd.DataFrame):
    
    for i, status in enumerate(raw_data['status']):

        if status == 'UNVALIDATED':
            rule_val = False

            for regla in lista_reglas:
                result = regla(raw_data)

                if result == True:
                    rule_val = result
                    break
                        
                else:
                    continue
            
            if rule_val == True:
                raw_data.at[i, 'status'] = 'USER-VALIDATION'
            
            else:
                pass
        
        else:
            pass
    


In [9]:
# Definición de funciones

def lectura_datos(url):
    url_acceso = url
    data = None
    if url_acceso.lower().endswith(".csv"):
       data = pd.read_csv(url_acceso, delimiter = ';', encoding = 'latin-1')
    
    elif url_acceso.lower().endswith(".txt"):
        data = pd.read_csv(url_acceso, delimiter = ';', encoding = 'latin-1')
    
    elif url_acceso.lower().endswith(".xlsx"):
        data = pd.read_excel(url_acceso)
    
    else:
      data = 'El archivo no es soportado, por favor cargar un archivo tipo csv, xlsx o txt'

    return data

In [10]:
prueba = lectura_datos(url)

C:\Users\juanm\AppData\Local\Temp\ipykernel_17196\908052247.py:7: DtypeWarning: Columns (1,19,30) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(url_acceso, delimiter = ';', encoding = 'latin-1')


In [11]:
prueba

,ric,rt_close,date,rt_ts,rt_currency,rt_trade_volume,region,mic,factset_close,factset_last,...,edi_updated_on,manual_close,manual_currency,manual_entry_date,manual_entry_source,trading_status,security_type,real_asset_class,golden_close,status
0,ARZA.KW,0.337,14/08/2025,2025-08-14 10:00:02.319749 UTC,KWD,10226,ASIA,XKUW,0.337,0.337,...,2025-08-15 22:04:34.880061 UTC,NaN,NaN,NaN,NaN,NORMAL,NaN,SHARE,0.337,VALIDATED
1,LVX.AS,0.9318,6/01/2025,2025-01-06 17:00:01.219813 UTC,EUR,50,EUROPE,XAMS,0.9318,0.9318,...,2025-01-08 23:01:57.681922 UTC,NaN,NaN,NaN,NaN,NORMAL,ETP,SHARE,0.9318,UNVALIDATED
2,CENCOSUD.SN,3140,17/06/2025,2025-06-17 20:02:07.214827 UTC,CLP,606,AMERICA,XSGO,3140,3140,...,2025-06-19 22:01:48.787615 UTC,NaN,NaN,NaN,NaN,NORMAL,Common Stock,SHARE,3140,VALIDATED
3,HHDC.CO,116,14/03/2025,2025-03-14 16:20:01.296614 UTC,DKK,100,EUROPE,XCSE,116,116,...,2025-03-16 23:01:52.013627 UTC,NaN,NaN,NaN,NaN,NORMAL,NaN,SHARE,116,VALIDATED
4,DSFIR.AS,97.72,31/12/2024,NaN,EUR,130,EUROPE,XAMS,97.72,97.72,...,2025-01-02 23:01:47.036302 UTC,NaN,NaN,NaN,NaN,NORMAL,Common Stock,SHARE,97.72,VALIDATED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28436,038070.KQ,7200.0,8/08/2025,2025-08-08 06:30:31.393484 UTC,KRW,979,ASIA,XKOS,7200,7200,...,2025-08-10 22:01:47.218802 UTC,NaN,NaN,NaN,NaN,NORMAL,Common Stock,SHARE,7200.0,VALIDATED
28437,240810.KQ,22250.0,20/12/2024,NaN,NaN,12432,ASIA,XKOS,22250,22250,...,2024-12-22 23:01:57.760949 UTC,NaN,NaN,NaN,NaN,NORMAL,Common Stock,SHARE,22250.0,VALIDATED
28438,450520.KQ,4955.0,18/02/2025,2025-02-18 06:30:31.785222 UTC,NaN,10591,ASIA,XKOS,4955,4955,...,2025-02-20 23:01:46.577762 UTC,NaN,NaN,NaN,NaN,NORMAL,NaN,SHARE,4955.0,VALIDATED
28439,102940.KQ,25000.0,13/12/2024,NaN,NaN,5956,ASIA,XKOS,25000,25000,...,2024-12-15 23:02:05.185419 UTC,NaN,NaN,NaN,NaN,NORMAL,Common Stock,SHARE,25000.0,VALIDATED


In [19]:
def comprobar_formato_precio(dataset):
    for i in ['rt_close', 'factset_close', 'factset_late_close', 'edi_close']:
        
        if dataset[i].dtype != float:
            dataset[i] = pd.to_numeric(dataset[i], errors='coerce')

    return dataset

In [20]:
comprobar_formato_precio(data)

,ric,rt_close,date,rt_ts,rt_currency,rt_trade_volume,region,mic,factset_close,factset_last,...,edi_updated_on,manual_close,manual_currency,manual_entry_date,manual_entry_source,trading_status,security_type,real_asset_class,golden_close,status
0,ARZA.KW,0.3370,14/08/2025,2025-08-14 10:00:02.319749 UTC,KWD,10226,ASIA,XKUW,0.3370,0.337,...,2025-08-15 22:04:34.880061 UTC,NaN,NaN,NaN,NaN,NORMAL,NaN,SHARE,0.337,VALIDATED
1,LVX.AS,0.9318,6/01/2025,2025-01-06 17:00:01.219813 UTC,EUR,50,EUROPE,XAMS,0.9318,0.9318,...,2025-01-08 23:01:57.681922 UTC,NaN,NaN,NaN,NaN,NORMAL,ETP,SHARE,0.9318,UNVALIDATED
2,CENCOSUD.SN,3140.0000,17/06/2025,2025-06-17 20:02:07.214827 UTC,CLP,606,AMERICA,XSGO,3140.0000,3140,...,2025-06-19 22:01:48.787615 UTC,NaN,NaN,NaN,NaN,NORMAL,Common Stock,SHARE,3140,VALIDATED
3,HHDC.CO,116.0000,14/03/2025,2025-03-14 16:20:01.296614 UTC,DKK,100,EUROPE,XCSE,116.0000,116,...,2025-03-16 23:01:52.013627 UTC,NaN,NaN,NaN,NaN,NORMAL,NaN,SHARE,116,VALIDATED
4,DSFIR.AS,97.7200,31/12/2024,NaN,EUR,130,EUROPE,XAMS,97.7200,97.72,...,2025-01-02 23:01:47.036302 UTC,NaN,NaN,NaN,NaN,NORMAL,Common Stock,SHARE,97.72,VALIDATED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28436,038070.KQ,7200.0000,8/08/2025,2025-08-08 06:30:31.393484 UTC,KRW,979,ASIA,XKOS,7200.0000,7200,...,2025-08-10 22:01:47.218802 UTC,NaN,NaN,NaN,NaN,NORMAL,Common Stock,SHARE,7200.0,VALIDATED
28437,240810.KQ,22250.0000,20/12/2024,NaN,NaN,12432,ASIA,XKOS,22250.0000,22250,...,2024-12-22 23:01:57.760949 UTC,NaN,NaN,NaN,NaN,NORMAL,Common Stock,SHARE,22250.0,VALIDATED
28438,450520.KQ,4955.0000,18/02/2025,2025-02-18 06:30:31.785222 UTC,NaN,10591,ASIA,XKOS,4955.0000,4955,...,2025-02-20 23:01:46.577762 UTC,NaN,NaN,NaN,NaN,NORMAL,NaN,SHARE,4955.0,VALIDATED
28439,102940.KQ,25000.0000,13/12/2024,NaN,NaN,5956,ASIA,XKOS,25000.0000,25000,...,2024-12-15 23:02:05.185419 UTC,NaN,NaN,NaN,NaN,NORMAL,Common Stock,SHARE,25000.0,VALIDATED


In [26]:
def exporte_depuracion(data, ruta_de_guardado, fecha_str):
    
    return data.to_csv(f'{ruta_de_guardado}/Export_Table_{fecha_str}.csv')

In [28]:
exporte_depuracion(data, "C:/Users/juanm/OneDrive/Documentos", '2025-10-19')

In [31]:
def corregir_denominaciones(data, umbral_superior=10, umbral_inferior=0.1, divisor=100):
    comparaciones = [
        ('rt_close', 'factset_close'),
        ('rt_close', 'factset_late_close'),
        ('rt_close', 'edi_close'),
        ('factset_close', 'edi_close'),
        ('factset_late_close', 'edi_close'),
    ]

    for i in range(data.shape[0]):  
        # Extraer divisas de forma segura
        rt_cur = data.at[i, 'rt_currency']
        fact_cur = data.at[i, 'factset_currency']
        edi_cur = data.at[i, 'edi_currency']

        # ⚠ Validar solo si todas tienen valor (no NaN o None)
        if pd.notna(rt_cur) and pd.notna(fact_cur) and pd.notna(edi_cur):
            # Si existen las 3 y no coinciden, alertar y saltar la fila
            if not (rt_cur == fact_cur == edi_cur):
                print(f"⚠ Problema de divisa en la fila {i}: rt={rt_cur}, factset={fact_cur}, edi={edi_cur}")
                continue  # No se corrige nada en esta fila
        
        # ✅ Si pasamos la validación, procedemos a corregir por denominación
        for base_col, check_col in comparaciones:
            base = data.at[i, base_col]
            check = data.at[i, check_col]

            if pd.isna(base) or pd.isna(check) or base == 0:
                continue

            ratio = check / base

            if ratio > umbral_superior:
                data.at[i, check_col] = check / divisor

            elif ratio < umbral_inferior:
                data.at[i, check_col] = check * divisor

    return data

In [32]:
corregir_denominaciones(data)

,ric,rt_close,date,rt_ts,rt_currency,rt_trade_volume,region,mic,factset_close,factset_last,...,edi_updated_on,manual_close,manual_currency,manual_entry_date,manual_entry_source,trading_status,security_type,real_asset_class,golden_close,status
0,ARZA.KW,0.3370,14/08/2025,2025-08-14 10:00:02.319749 UTC,KWD,10226,ASIA,XKUW,0.3370,0.337,...,2025-08-15 22:04:34.880061 UTC,NaN,NaN,NaN,NaN,NORMAL,NaN,SHARE,0.337,VALIDATED
1,LVX.AS,0.9318,6/01/2025,2025-01-06 17:00:01.219813 UTC,EUR,50,EUROPE,XAMS,0.9318,0.9318,...,2025-01-08 23:01:57.681922 UTC,NaN,NaN,NaN,NaN,NORMAL,ETP,SHARE,0.9318,UNVALIDATED
2,CENCOSUD.SN,3140.0000,17/06/2025,2025-06-17 20:02:07.214827 UTC,CLP,606,AMERICA,XSGO,3140.0000,3140,...,2025-06-19 22:01:48.787615 UTC,NaN,NaN,NaN,NaN,NORMAL,Common Stock,SHARE,3140,VALIDATED
3,HHDC.CO,116.0000,14/03/2025,2025-03-14 16:20:01.296614 UTC,DKK,100,EUROPE,XCSE,116.0000,116,...,2025-03-16 23:01:52.013627 UTC,NaN,NaN,NaN,NaN,NORMAL,NaN,SHARE,116,VALIDATED
4,DSFIR.AS,97.7200,31/12/2024,NaN,EUR,130,EUROPE,XAMS,97.7200,97.72,...,2025-01-02 23:01:47.036302 UTC,NaN,NaN,NaN,NaN,NORMAL,Common Stock,SHARE,97.72,VALIDATED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28436,038070.KQ,7200.0000,8/08/2025,2025-08-08 06:30:31.393484 UTC,KRW,979,ASIA,XKOS,7200.0000,7200,...,2025-08-10 22:01:47.218802 UTC,NaN,NaN,NaN,NaN,NORMAL,Common Stock,SHARE,7200.0,VALIDATED
28437,240810.KQ,22250.0000,20/12/2024,NaN,NaN,12432,ASIA,XKOS,22250.0000,22250,...,2024-12-22 23:01:57.760949 UTC,NaN,NaN,NaN,NaN,NORMAL,Common Stock,SHARE,22250.0,VALIDATED
28438,450520.KQ,4955.0000,18/02/2025,2025-02-18 06:30:31.785222 UTC,NaN,10591,ASIA,XKOS,4955.0000,4955,...,2025-02-20 23:01:46.577762 UTC,NaN,NaN,NaN,NaN,NORMAL,NaN,SHARE,4955.0,VALIDATED
28439,102940.KQ,25000.0000,13/12/2024,NaN,NaN,5956,ASIA,XKOS,25000.0000,25000,...,2024-12-15 23:02:05.185419 UTC,NaN,NaN,NaN,NaN,NORMAL,Common Stock,SHARE,25000.0,VALIDATED


In [33]:
def consulta_csv(ruta):
    data = pd.read_csv(ruta)

    return data

In [34]:
def añadir_regla(regla):
    lista_reglas.append(regla)